In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as f
from pyspark.sql import types as t
from pyspark.sql.window import Window
from datetime import datetime
import logging        
from config import ROUTES, PipelineConfig  

In [0]:
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
log = logging.getLogger(__name__)

In [0]:
# Config 
BRONZE_PATH = "workspace.case_spark_cvm.bronze_fii_ativo_passivo_cvm"
NOME_TABELA  = f"silver_cvm_fii_ativo_passivo" 
SILVER_PATH = f"{ROUTES.TABLE_BASE}.{NOME_TABELA}"
DATA_PROC    = int(datetime.now().strftime("%Y%m%d"))

## CVM - Fundos Imobiliarios - Ativo Passivo

In [0]:
df_silver_fii_ativo_passivo = PipelineConfig.ler_ultima_particao(spark=spark, table_name=BRONZE_PATH, partition_col="data_processamento" )

### 1.1 tratemento silver

#### 1.1.1 Normalizando CNPJ

In [0]:
df_silver_fii_ativo_passivo = df_silver_fii_ativo_passivo.withColumn(
    "CNPJ_FUNDO_CLASSE",
    PipelineConfig.normalizar_cnpj("CNPJ_FUNDO_CLASSE")
)


#### 1.1.2 Retirando dados duplicados

In [0]:
# 1. Chaves que devem ser únicas
chave_negocio = ["CNPJ_FUNDO_CLASSE", "Data_Referencia"]

# 2. Removemos as duplicatas 
# Como não á uma regra clara de desempate ficamos com a linha com maior patrimonio liquido
df_silver_fii_ativo_passivo, df_quarentena_duplicadas = PipelineConfig.remover_duplicatas(
    df=df_silver_fii_ativo_passivo,
    chave_negocio=chave_negocio,
    coluna_ordenacao="Total_Investido" 
)

# 3. Salva a sujeira na quarentena
PipelineConfig.salvar_quarentena(
    spark=spark,
    df_quarentena=df_quarentena_duplicadas, 
    tabela_origem="bronze_fii_ativo_passivo_cvm", 
    data_proc=DATA_PROC
)


#### 1.1.3 Retirando dados nulos de Colunas Cores

In [0]:
regras_qualidade = {
    "CNPJ_FUNDO_CLASSE": "not_null",  # Não pode ser vazio (Substitui o dropna)
    "Data_Referencia": "not_null",    # Não pode ser vazio (Substitui o dropna)
    "Total_Necessidades_Liquidez": "decimal",         # Não pode conter letras
    "Disponibilidades": "decimal",  # Não pode conter letras
    "Total_Investido": "decimal"   # Não pode conter letras
}

df_silver_fii_ativo_passivo, df_quarentena = PipelineConfig.aplicar_qualidade_e_separar(
    df=df_silver_fii_ativo_passivo,
    regras=regras_qualidade
    )

PipelineConfig.salvar_quarentena(
    spark=spark,
    df_quarentena=df_quarentena, 
    tabela_origem="bronze_fii_ativo_passivo_cvm", 
    data_proc=DATA_PROC
)

#### 1.1.4 Tratamento do Tipo de Dado

In [0]:
# Dropando as colunas de metadados
df_silver_fii_ativo_passivo = df_silver_fii_ativo_passivo.drop("_source_url", "_ingest_timestamp", "data_processamento")


In [0]:
# ==============================================================================
# SELEÇÃO E CASTING FINAL (Otimizado via Select único - Catalyst Optimizer)
# ==============================================================================

df_silver_fii_ativo_passivo = df_silver_fii_ativo_passivo.select(
    # Se quiser aplicar a sua função de CNPJ aqui em vez do cast simples, 
    # basta trocar para: PipelineConfig.normalizar_cnpj('CNPJ_FUNDO_CLASSE').alias('cnpj_fundo_classe')
    f.col('CNPJ_FUNDO_CLASSE').cast(t.StringType()).alias('cnpj_fundo_classe'),
    f.col('Data_Referencia').cast(t.DateType()).alias('data_referencia'),
    f.col('Versao').cast(t.IntegerType()).alias('versao'),
    
    # Bloco de Ativos e Passivos
    f.col('Total_Necessidades_Liquidez').cast(t.DecimalType(22, 2)).alias('total_necessidades_liquidez'),
    f.col('Disponibilidades').cast(t.DecimalType(22, 2)).alias('disponibilidades'),
    f.col('Titulos_Publicos').cast(t.DecimalType(22, 2)).alias('titulos_publicos'),
    f.col('Titulos_Privados').cast(t.DecimalType(22, 2)).alias('titulos_privados'),
    f.col('Fundos_Renda_Fixa').cast(t.DecimalType(18, 2)).alias('fundos_renda_fixa'),
    f.col('Total_Investido').cast(t.DecimalType(18, 2)).alias('total_investido'),
    f.col('Direitos_Bens_Imoveis').cast(t.DecimalType(22, 2)).alias('direitos_bens_imoveis'),
    f.col('Terrenos').cast(t.DecimalType(18, 2)).alias('terrenos'),
    f.col('Imoveis_Renda_Acabados').cast(t.DecimalType(22, 2)).alias('imoveis_renda_acabados'),
    f.col('Imoveis_Renda_Construcao').cast(t.DecimalType(22, 2)).alias('imoveis_renda_construcao'),
    f.col('Imoveis_Venda_Acabados').cast(t.DecimalType(22, 2)).alias('imoveis_venda_acabados'),
    f.col('Imoveis_Venda_Construcao').cast(t.DecimalType(22, 2)).alias('imoveis_venda_construcao'),
    f.col('Outros_Direitos_Reais').cast(t.DecimalType(22, 2)).alias('outros_direitos_reais'),
    f.col('Acoes').cast(t.DecimalType(22, 2)).alias('acoes'),
    f.col('Debentures').cast(t.DecimalType(22, 2)).alias('debentures'),
    f.col('Bonus_Subscricao').cast(t.DecimalType(22, 2)).alias('bonus_subscricao'),
    f.col('Certificados_Deposito_Valores_Mobiliarios').cast(t.DecimalType(22, 2)).alias('certificados_deposito_valores_mobiliarios'),
    f.col('Cedulas_Debentures').cast(t.DecimalType(12, 2)).alias('cedulas_debentures'),
    f.col('Fundo_Acoes').cast(t.DecimalType(22, 2)).alias('fundo_acoes'),
    f.col('FIP').cast(t.DecimalType(22, 2)).alias('fip'),
    f.col('FII').cast(t.DecimalType(22, 2)).alias('fii'),
    f.col('FDIC').cast(t.DecimalType(22, 2)).alias('fdic'),
    f.col('Outras_Cotas_FI').cast(t.DecimalType(22, 2)).alias('outras_cotas_fi'),
    f.col('Notas_Promissorias').cast(t.DecimalType(22, 2)).alias('notas_promissorias'),
    f.col('Acoes_Sociedades_Atividades_FII').cast(t.DecimalType(22, 2)).alias('acoes_sociedades_atividades_fii'),
    f.col('Cotas_Sociedades_Atividades_FII').cast(t.DecimalType(22, 2)).alias('cotas_sociedades_atividades_fii'),
    f.col('CEPAC').cast(t.DecimalType(22, 2)).alias('cepac'),
    f.col('CRI').cast(t.DecimalType(22, 2)).alias('cri'),
    f.col('CRI_CRA').cast(t.DecimalType(22, 2)).alias('cri_cra'),
    f.col('Letras_Hipotecarias').cast(t.DecimalType(22, 2)).alias('letras_hipotecarias'),
    f.col('LCI').cast(t.DecimalType(22, 2)).alias('lci'),
    f.col('LCI_LCA').cast(t.DecimalType(22, 2)).alias('lci_lca'),
    f.col('LIG').cast(t.DecimalType(22, 2)).alias('lig'),
    f.col('Outros_Valores_Mobliarios').cast(t.DecimalType(22, 2)).alias('outros_valores_mobliarios'),
    f.col('Valores_Receber').cast(t.DecimalType(22, 2)).alias('valores_receber'),
    f.col('Contas_Receber_Aluguel').cast(t.DecimalType(22, 2)).alias('contas_receber_aluguel'),
    f.col('Contas_Receber_Venda_Imoveis').cast(t.DecimalType(22, 2)).alias('contas_receber_venda_imoveis'),
    f.col('Outros_Valores_Receber').cast(t.DecimalType(22, 2)).alias('outros_valores_receber'),
    f.col('Rendimentos_Distribuir').cast(t.DecimalType(22, 2)).alias('rendimentos_distribuir'),
    f.col('Taxa_Administracao_Pagar').cast(t.DecimalType(22, 2)).alias('taxa_administracao_pagar'),
    f.col('Taxa_Performance_Pagar').cast(t.DecimalType(22, 2)).alias('taxa_performance_pagar'),
    f.col('Obrigacoes_Aquisicao_Imoveis').cast(t.DecimalType(22, 2)).alias('obrigacoes_aquisicao_imoveis'),
    f.col('Adiantamento_Venda_Imoveis').cast(t.DecimalType(22, 2)).alias('adiantamento_venda_imoveis'),
    f.col('Adiantamento_Alugueis').cast(t.DecimalType(22, 2)).alias('adiantamento_alugueis'),
    f.col('Obrigacoes_Securitizacao_Recebiveis').cast(t.DecimalType(22, 2)).alias('obrigacoes_securitizacao_recebiveis'),
    f.col('Instrumentos_Financeiros_Derivativos').cast(t.DecimalType(22, 2)).alias('instrumentos_financeiros_derivativos'),
    f.col('Provisoes_Contigencias').cast(t.DecimalType(22, 2)).alias('provisoes_contigencias'),
    f.col('Outros_Valores_Pagar').cast(t.DecimalType(22, 2)).alias('outros_valores_pagar'),
    f.col('Total_Passivo').cast(t.DecimalType(22, 2)).alias('total_passivo')
)

### 1.2 Salvar na camada Silver

In [0]:
# Definindo as chaves estrangeiras 
chave_negocio = ["cnpj_fundo_classe", "data_referencia"]

PipelineConfig.upsert_silver(
    spark=spark, 
    df_novo=df_silver_fii_ativo_passivo, 
    tabela_destino=SILVER_PATH, 
    chave_negocio=chave_negocio
    )